In [1]:
import pandas as pd

df = pd.read_csv("../data/clean_orders.csv", parse_dates=["order_purchase_timestamp"])
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,price,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,72632f0f9dd73dfee390c9b22eb56dd6


In [2]:
snapshot_date = df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

Snapshot date: 2018-08-30 15:00:37


In [3]:
rfm = df.groupby("customer_unique_id").agg(
    recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("price", "sum")
).reset_index()

print(rfm.shape)
rfm.head()

(93358, 4)


,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,18.90
2,0000f46a3911fa3c0805444483337064,537,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,288,1,180.00


In [4]:
rfm["recency"].describe()

count    93358.000000
mean       237.941773
std        152.591453
min          1.000000
25%        114.000000
50%        219.000000
75%        346.000000
max        714.000000
Name: recency, dtype: float64

In [5]:
rfm["churned"] = (rfm["recency"] > 180).astype(int)

print(rfm["churned"].value_counts())
print(rfm["churned"].value_counts(normalize=True))

churned
1    55252
0    38106
Name: count, dtype: int64
churned
1    0.591829
0    0.408171
Name: proportion, dtype: float64


In [6]:
rfm.groupby("churned")[["recency", "frequency", "monetary"]].mean()

,recency,frequency,monetary
churned,,,
0,92.134861,1.037369,144.441984
1,338.501357,1.030696,139.676244


In [7]:
rfm.to_csv("../data/rfm_churn.csv", index=False)
print("Saved rfm_churn.csv")

Saved rfm_churn.csv


## Churn Definition
- Churn defined as: no purchase in the last 180 days (6 months) from snapshot date
- Chosen based on recency distribution — median recency was [fill in your number] days,
  so 180 days captures meaningfully inactive customers rather than a random cutoff
- Resulting split: [fill in]% active vs [fill in]% churned
- RFM (Recency, Frequency, Monetary) computed per unique customer using customer_unique_id